# Mathematical Derivations of Backpropagation
### *backprop_from_bedrock*

This notebook provides rigorous mathematical derivations for every backward pass implemented in the library, along with interactive verification.


## 1. Linear (Dense) Layer & Matrix Multiplication

Let $X \in \mathbb{R}^{B \times D_{in}}$ be the input batch, $W \in \mathbb{R}^{D_{in} \times D_{out}}$ the weight matrix, and $b \in \mathbb{R}^{D_{out}}$ the bias vector.

The forward pass is:
$$Z = X W + b \quad \in \mathbb{R}^{B \times D_{out}}$$

Let upstream gradient from scalar loss $L$ be $dZ = \frac{\partial L}{\partial Z} \in \mathbb{R}^{B \times D_{out}}$.

Using matrix calculus:
- $\frac{\partial L}{\partial X} = dZ \cdot W^T \in \mathbb{R}^{B \times D_{in}}$
- $\frac{\partial L}{\partial W} = X^T \cdot dZ \in \mathbb{R}^{D_{in} \times D_{out}}$
- $\frac{\partial L}{\partial b} = \sum_{i=1}^B dZ_{i, :} \in \mathbb{R}^{D_{out}}$


In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
from core import Tensor, Linear

X = Tensor(np.random.randn(5, 4), requires_grad=True)
linear = Linear(4, 3)
out = linear(X).sum()
out.backward()
print("X grad shape:", X.grad.shape)
print("W grad shape:", linear.weight.grad.shape)
print("b grad shape:", linear.bias.grad.shape)


X grad shape: (5, 4)
W grad shape: (4, 3)
b grad shape: (3,)


## 2. Activation Functions

### ReLU
$$y = \max(0, x) \implies \frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot \mathbf{1}_{\{x > 0\}}$$

### Sigmoid
$$\sigma(x) = \frac{1}{1 + e^{-x}} \implies \frac{\partial \sigma}{\partial x} = \sigma(x)(1 - \sigma(x))$$
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot y \odot (1 - y)$$

### Tanh
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} \implies \frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \odot (1 - \tanh^2(x))$$

### Softmax
$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}} \implies \frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j)$$
$$\frac{\partial L}{\partial z_i} = p_i \left( \frac{\partial L}{\partial p_i} - \sum_j \frac{\partial L}{\partial p_j} p_j \right)$$


In [2]:
x = Tensor(np.array([-2.0, 0.0, 1.5, 3.0]), requires_grad=True)
y = x.relu().sum()
y.backward()
print("ReLU grad:", x.grad)


ReLU grad: [0. 0. 1. 1.]


## 3. Cross-Entropy Loss with Softmax

Given logits $z \in \mathbb{R}^{B \times C}$ and true class indices $y_i \in \{0, \dots, C-1\}$:
$$L = -\frac{1}{B} \sum_{i=1}^B \log p_{i, y_i} = -\frac{1}{B} \sum_{i=1}^B \left( z_{i, y_i} - \log \sum_{j=1}^C e^{z_{ij}} \right)$$

Differentiating w.r.t logits $z_{ic}$:
$$\frac{\partial L}{\partial z_{ic}} = \frac{1}{B} (p_{ic} - \mathbf{1}_{\{c = y_i\}})$$

This achieves extreme numerical stability when computed directly from shifted logits (Log-Sum-Exp trick).


In [3]:
from core import CrossEntropyLoss

logits = Tensor(np.random.randn(4, 3), requires_grad=True)
targets = np.array([0, 2, 1, 0])
loss_fn = CrossEntropyLoss()
loss = loss_fn(logits, targets)
loss.backward()
print("Loss value:", loss.item())
print("Logits grad shape:", logits.grad.shape)


Loss value: 1.9160799034080949
Logits grad shape: (4, 3)
